# TRANSFORMACIÓN DE DATOS

## IMPORTAR PAQUETES

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import janitor
import os
from pathlib import Path
import openpyxl
import sqlalchemy as sa

# Desactivar notación científica
pd.set_option('display.float_format', lambda x: '%.3f' % x)
np.set_printoptions(suppress=True)

# Cargar variables de entorno
load_dotenv()

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


## IMPORTAR LOS DATOS

In [2]:
num = pd.read_pickle('../02_datos/03_Entrenamiento/num_eda.pkl')
cat = pd.read_pickle('../02_datos/03_Entrenamiento/cat_eda.pkl')

Recordamos los que habíamos identificado en fases anteriores y lo organizamos por tipo de acción

Modificar con pandas/numpy:

- Crear la target apartir de estado
- Discretizar num_meses_desde_ult_retraso
- En vivienda juntar ANY, NONE Y OTHER
- En finalidad juntar renewable_energy, educational y wedding en 'otros'

Modificar con scikit-learn

- Convertir num_derogatorios a binaria

## CREACIÓN DE VARIABLES Y TRANSFORMACIONES CON PANDAS

### Target para el modelo PD

Crear la target apartir de estado

In [3]:
cat.estado.value_counts()

estado
Fully Paid                                             66598
Current                                                54220
Charged Off                                            16445
Late (31-120 days)                                      1362
In Grace Period                                          530
Late (16-30 days)                                        280
Does not meet the credit policy. Status:Fully Paid       114
Does not meet the credit policy. Status:Charged Off       53
Default                                                    2
Name: count, dtype: int64

In [4]:
cat['target_pd'] = np.where(cat.estado.isin(['Charged Off', 'Does not meet the credit policy. Status:Charged Off', 'Default']), 1, 0)
cat

,antigüedad_empleo,ingresos_verificados,rating,vivienda,finalidad,num_cuotas,estado,target_pd
0,3 years,Source Verified,A,MORTGAGE,debt_consolidation,36 months,Current,0
1,5 years,Verified,A,RENT,debt_consolidation,36 months,Fully Paid,0
2,9 years,Source Verified,B,MORTGAGE,credit_card,60 months,Current,0
3,7 years,Source Verified,B,MORTGAGE,debt_consolidation,36 months,Fully Paid,0
4,10+ years,Source Verified,D,MORTGAGE,home_improvement,60 months,Current,0
...,...,...,...,...,...,...,...,...
139995,10+ years,Verified,B,RENT,debt_consolidation,36 months,Fully Paid,0
139996,desconocido,Source Verified,C,RENT,debt_consolidation,36 months,Late (31-120 days),0
139997,10+ years,Not Verified,C,MORTGAGE,debt_consolidation,36 months,Fully Paid,0
139998,6 years,Verified,B,MORTGAGE,other,36 months,Current,0


In [5]:
cat.drop(columns=['estado'], inplace=True)

In [6]:
num.rename(columns={'dti': 'DTI'}, inplace=True)

### Target para el modelo EAD

Será el porcentaje del principal que no haya sido amortizado.

Por tanto: pendiente / principal

Primero tenemos que construir el pendiente como el principal menos el amortizado.

In [7]:
num['pendiente'] = num['principal'] - num['imp_amortizado']

In [8]:
num['target_ead'] = num['pendiente'] / num['principal']
num

,id_cliente,ingresos,DTI,num_hipotecas,num_lineas_credito,porc_tarjetas_75p,porc_uso_revolving,num_cancelaciones_12meses,num_derogatorios,num_meses_desde_ult_retraso,principal,tipo_interes,imp_cuota,imp_amortizado,imp_recuperado,pendiente,target_ead
0,137387967,54000.000,19.310,2,10,33.300,45.200,0,0,10,15000.000,7.210,464.600,2669.060,0.000,12330.940,0.822
1,46641215,135000.000,14.680,0,19,0.000,16.300,0,0,0,24000.000,6.390,734.380,24000.000,0.000,0.000,0.000
2,87998444,188000.000,11.690,3,15,0.000,3.300,0,0,0,27000.000,8.990,560.350,12443.000,0.000,14557.000,0.539
3,132883631,125000.000,9.000,1,6,33.300,69.900,0,0,16,22000.000,10.900,719.220,22000.000,0.000,0.000,0.000
4,131289518,72777.000,24.740,2,10,33.300,63.200,0,0,0,25000.000,21.850,688.350,2811.270,0.000,22188.730,0.888
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139995,73222117,85000.000,33.050,1,25,90.000,76.000,1,0,0,3000.000,9.750,96.450,3000.000,0.000,0.000,0.000
139996,119242742,20000.000,8.020,0,5,66.700,33.200,0,0,0,2500.000,12.620,83.780,996.060,0.000,1503.940,0.602
139997,51876926,42000.000,20.850,6,9,100.000,81.600,0,0,13,8000.000,12.290,266.830,8000.000,0.000,0.000,0.000
139998,121031962,111697.000,16.630,2,10,50.000,26.300,0,1,6,10000.000,9.440,320.050,4388.510,0.000,5611.490,0.561


### Target para el modelo LGD

Será el porcentaje del pendiente que NO se recupera en caso de impago

Por tanto: 1 - (imp_recuperado / pendiente)

In [13]:
num['target_lgd'] = 1 - (num['imp_recuperado'] / num['pendiente'])
num

,id_cliente,ingresos,DTI,num_hipotecas,num_lineas_credito,porc_tarjetas_75p,porc_uso_revolving,num_cancelaciones_12meses,num_derogatorios,num_meses_desde_ult_retraso,principal,tipo_interes,imp_cuota,imp_amortizado,imp_recuperado,pendiente,target_ead,target_lgd
0,137387967,54000.000,19.310,2,10,33.300,45.200,0,0,10,15000.000,7.210,464.600,2669.060,0.000,12330.940,0.822,1.000
1,46641215,135000.000,14.680,0,19,0.000,16.300,0,0,0,24000.000,6.390,734.380,24000.000,0.000,0.000,0.000,NaN
2,87998444,188000.000,11.690,3,15,0.000,3.300,0,0,0,27000.000,8.990,560.350,12443.000,0.000,14557.000,0.539,1.000
3,132883631,125000.000,9.000,1,6,33.300,69.900,0,0,16,22000.000,10.900,719.220,22000.000,0.000,0.000,0.000,NaN
4,131289518,72777.000,24.740,2,10,33.300,63.200,0,0,0,25000.000,21.850,688.350,2811.270,0.000,22188.730,0.888,1.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139995,73222117,85000.000,33.050,1,25,90.000,76.000,1,0,0,3000.000,9.750,96.450,3000.000,0.000,0.000,0.000,NaN
139996,119242742,20000.000,8.020,0,5,66.700,33.200,0,0,0,2500.000,12.620,83.780,996.060,0.000,1503.940,0.602,1.000
139997,51876926,42000.000,20.850,6,9,100.000,81.600,0,0,13,8000.000,12.290,266.830,8000.000,0.000,0.000,0.000,NaN
139998,121031962,111697.000,16.630,2,10,50.000,26.300,0,1,6,10000.000,9.440,320.050,4388.510,0.000,5611.490,0.561,1.000


Los nulos se producen cuando el pendiente es cero. Por tanto podemos imputarlos por ceros

In [14]:
num['target_lgd'] = num['target_lgd'].fillna(0)

> ⚠️ **OBSERVACIONES PENDIENTES SOBRE target_lgd Y target_ead** (revisar más adelante)
>
> **1. LGD negativas (58 casos).** En algunos préstamos imp_recuperado > pendiente (p. ej. fila 313: pendiente 9.795 vs recuperado 10.101), lo que da LGD de hasta -0,28. La LGD debe estar en [0, 1]:

> **2. Pendiente negativo (13 casos).** Ajustes de céntimos (amortizado 7.800,25 sobre principal 7.800 → pendiente -0,25). Afecta también a target_ead. Recortar antes de calcular ambas targets:

Comprobar las celdas creadas

In [15]:
cat[['target_pd']].describe().T

,count,mean,std,min,25%,50%,75%,max
target_pd,139604.000,0.118,0.323,0.000,0.000,0.000,0.000,1.000


In [16]:
num[['target_ead', 'target_lgd']].describe().T

,count,mean,std,min,25%,50%,75%,max
target_ead,139604.000,0.337,0.370,-0.000,0.000,0.126,0.726,1.000
target_lgd,139604.000,0.508,0.490,-0.276,0.000,0.848,1.000,1.000


Limitamos para garantizar que el rango este siempre entre 0 y 1

In [18]:
num.target_ead = np.where(num.target_ead < 0, 0, num.target_ead)
num.target_ead = np.where(num.target_ead > 1, 1, num.target_ead)
num.target_lgd = np.where(num.target_lgd < 0, 0, num.target_lgd)
num.target_lgd = np.where(num.target_lgd > 1, 1, num.target_lgd)

In [19]:
num[['target_ead', 'target_lgd']].describe().T

,count,mean,std,min,25%,50%,75%,max
target_ead,139604.000,0.337,0.370,0.000,0.000,0.126,0.726,1.000
target_lgd,139604.000,0.508,0.490,0.000,0.000,0.848,1.000,1.000


### Resto de Variables